# 01 · Feasibility spike

Check the data, build the suites, measure the reward's blind spot **before any training**, and baseline candidate models. Everything here is cheap; it decides the rest of the plan.

In [ ]:
# --- setup: run once per session -------------------------------------------
import os, subprocess, sys
SPIDER_DIR = "/kaggle/input/spider/spider"   # <- EDIT: the folder with train_spider.json and database/
RUNS = "/kaggle/working/runs"
os.environ["SPIDER_DIR"], os.environ["MIRAGE_RUNS"] = SPIDER_DIR, RUNS
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
os.chdir("/kaggle/working/mirage")
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Two-minute smoke test
Runs every TRL/PEFT call in the training code on a tiny random model. If a library update broke something, you find out now, not two hours into a run.

In [ ]:
!python scripts/smoke_train.py --device cuda --out /kaggle/working/smoke

In [ ]:
!mirage data check $SPIDER_DIR

## The blind spot, before training (CPU only)
Building suites for all ~200 Spider databases takes a while on CPU; start with a subset to see the numbers, then build all of them once.

In [ ]:
!mirage suites build --spider $SPIDER_DIR --out $MIRAGE_RUNS/suites --only concert_singer,pets_1,car_1,flight_2,world_1

In [ ]:
# all databases (train + dev); run once, keep the output
!mirage suites build --spider $SPIDER_DIR --out $MIRAGE_RUNS/suites

## Baseline the candidates on a val slice
Want: EX roughly 15–50%, and many questions where the 8 attempts disagree ('mixed groups'). Near 0% means an SFT warm-up first.

In [ ]:
!mirage spike -c configs/spike.yaml --n 100

Write the decision down: which model, and whether to warm up. Then edit `configs/base.yaml` (`model.name`) and, if skipping SFT, remove `init_adapter` from `configs/grpo_base.yaml`.